In [1]:

import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ==========================================================
# 1. PROJECT CONFIGURATION
# ==========================================================

# Your specified dataset path
DATASET_DIR = Path(
    r"C:\Users\khare\OneDrive\Desktop\project\archive"
)

DOC_PATH = DATASET_DIR / "CISI.ALL"
QRY_PATH = DATASET_DIR / "CISI.QRY"
REL_PATH = DATASET_DIR / "CISI.REL"

# Download required NLTK resources
nltk.download("stopwords", quiet=True)

STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()


# ==========================================================
# 2. DATASET PARSING MODULE
# ==========================================================

def parse_cisi_all(filepath):
    """Parse document IDs, titles, and abstracts from CISI.ALL."""

    documents = {}
    current_doc = None
    current_field = None

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line_stripped = line.strip()

            if line_stripped.startswith(".I"):
                if current_doc is not None:
                    content = (
                        current_doc["title"] + " " +
                        current_doc["text"]
                    ).strip()

                    documents[current_doc["id"]] = content

                current_doc = {
                    "id": line_stripped[2:].strip(),
                    "title": "",
                    "text": ""
                }
                current_field = None

            elif current_doc is not None and line_stripped.startswith(".T"):
                current_field = "title"
                current_doc["title"] += line_stripped[2:].strip() + " "

            elif current_doc is not None and line_stripped.startswith(".A"):
                current_field = None

            elif current_doc is not None and line_stripped.startswith(".W"):
                current_field = "text"
                current_doc["text"] += line_stripped[2:].strip() + " "

            elif current_doc is not None and line_stripped.startswith(".X"):
                current_field = None

            elif current_doc is not None and current_field in ("title", "text"):
                current_doc[current_field] += line_stripped + " "

    # Save the final document
    if current_doc is not None:
        content = (
            current_doc["title"] + " " +
            current_doc["text"]
        ).strip()

        documents[current_doc["id"]] = content

    return documents


def parse_cisi_qry(filepath):
    """Parse query IDs and query text from CISI.QRY."""

    queries = {}
    current_query = None
    current_field = None

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line_stripped = line.strip()

            if line_stripped.startswith(".I"):
                if current_query is not None:
                    queries[current_query["id"]] = (
                        current_query["text"].strip()
                    )

                current_query = {
                    "id": line_stripped[2:].strip(),
                    "text": ""
                }
                current_field = None

            elif current_query is not None and line_stripped.startswith(".W"):
                current_field = "text"
                current_query["text"] += line_stripped[2:].strip() + " "

            elif current_query is not None and current_field == "text":
                current_query["text"] += line_stripped + " "

    # Save the final query
    if current_query is not None:
        queries[current_query["id"]] = current_query["text"].strip()

    return queries


def parse_cisi_rel(filepath):
    """Parse relevance judgments from CISI.REL."""

    relevance = defaultdict(set)

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.split()

            if len(parts) >= 2:
                query_id = parts[0]
                document_id = parts[1]
                relevance[query_id].add(document_id)

    return relevance


# ==========================================================
# 3. TEXT PREPROCESSING MODULE
# ==========================================================

def preprocess_text(text):
    """
    Apply lowercasing, tokenization, stopword removal,
    and Porter stemming.
    """

    text = text.lower()
    tokens = re.findall(r"\b\w+\b", text)

    # Remove English stopwords
    tokens = [
        token for token in tokens
        if token not in STOP_WORDS
    ]

    # Apply stemming
    tokens = [
        STEMMER.stem(token)
        for token in tokens
    ]

    return " ".join(tokens)


# ==========================================================
# 4. INDEXING AND RETRIEVAL MODEL
# ==========================================================

class InformationRetrievalSystem:

    def __init__(self, documents):
        self.documents = documents
        self.doc_ids = list(documents.keys())

        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = None
        self.is_indexed = False

    def build_index(self):
        """Build a TF-IDF index for all documents."""

        print("\nBuilding TF-IDF index...")

        processed_docs = [
            preprocess_text(self.documents[doc_id])
            for doc_id in self.doc_ids
        ]

        if not processed_docs or not any(processed_docs):
            raise ValueError(
                "No usable document text found in CISI.ALL."
            )

        self.tfidf_matrix = self.vectorizer.fit_transform(
            processed_docs
        )

        self.is_indexed = True

        print("TF-IDF index built successfully.")
        print(f"Total documents indexed: {len(self.doc_ids)}")
        print(
            f"Unique terms indexed: "
            f"{len(self.vectorizer.vocabulary_)}"
        )

    def search(self, query, top_k=5):
        """Retrieve the top-k documents using cosine similarity."""

        if not self.is_indexed:
            raise RuntimeError(
                "Index not built. Call build_index() first."
            )

        processed_query = preprocess_text(query)

        if not processed_query:
            return []

        query_vector = self.vectorizer.transform(
            [processed_query]
        )

        similarities = cosine_similarity(
            query_vector,
            self.tfidf_matrix
        ).flatten()

        # Rank documents by descending similarity
        top_indices = np.argsort(similarities)[::-1]

        results = []

        for idx in top_indices:
            score = float(similarities[idx])

            # Exclude documents with zero similarity
            if score <= 0:
                continue

            doc_id = self.doc_ids[idx]
            original_text = self.documents[doc_id]

            snippet = original_text[:200]

            if len(original_text) > 200:
                snippet += "..."

            results.append({
                "doc_id": doc_id,
                "score": score,
                "snippet": snippet
            })

            if len(results) >= top_k:
                break

        return results


# ==========================================================
# 5. EVALUATION MODULE
# ==========================================================

def evaluate_system(ir_system, queries, relevance, top_k=5):
    """Calculate mean Precision@K and Recall@K."""

    precisions = []
    recalls = []

    for query_id, query_text in queries.items():

        relevant_docs = relevance.get(query_id, set())

        if not relevant_docs:
            continue

        results = ir_system.search(
            query_text,
            top_k=top_k
        )

        retrieved_docs = {
            result["doc_id"]
            for result in results
        }

        true_positives = len(
            relevant_docs.intersection(retrieved_docs)
        )

        # Precision@K: relevant results divided by K
        precision = true_positives / top_k

        # Recall@K: relevant results retrieved divided by
        # all known relevant documents for the query
        recall = true_positives / len(relevant_docs)

        precisions.append(precision)
        recalls.append(recall)

    if not precisions:
        return 0.0, 0.0

    return (
        float(np.mean(precisions)),
        float(np.mean(recalls))
    )


# ==========================================================
# 6. MAIN EXECUTION
# ==========================================================

def main():

    print("=" * 70)
    print("CISI INFORMATION RETRIEVAL SYSTEM")
    print("=" * 70)

    print(f"\nDataset folder: {DATASET_DIR}")

    # ------------------------------------------------------
    # CHECK DATASET FILES
    # ------------------------------------------------------

    required_files = [DOC_PATH, QRY_PATH, REL_PATH]

    missing_files = [
        path for path in required_files
        if not path.is_file()
    ]

    if missing_files:
        print("\nERROR: Dataset files not found:")

        for path in missing_files:
            print(f"  {path}")

        print(
            "\nPlease extract CISI.ALL, CISI.QRY, and CISI.REL "
            "into the archive folder."
        )

        return

    # ------------------------------------------------------
    # LOAD DATASET
    # ------------------------------------------------------

    print("\nLoading CISI datasets...")

    documents = parse_cisi_all(DOC_PATH)
    queries = parse_cisi_qry(QRY_PATH)
    relevance = parse_cisi_rel(REL_PATH)

    print(f"Documents loaded: {len(documents)}")
    print(f"Queries loaded: {len(queries)}")
    print(
        f"Queries with relevance judgments: {len(relevance)}"
    )

    if not documents:
        print("No documents were parsed. Check CISI.ALL.")
        return

    if not queries:
        print("No queries were parsed. Check CISI.QRY.")
        return

    # ------------------------------------------------------
    # BUILD INDEX
    # ------------------------------------------------------

    ir_system = InformationRetrievalSystem(documents)
    ir_system.build_index()

    # ------------------------------------------------------
    # PART A: AUTOMATED QUERY TESTING
    # ------------------------------------------------------

    print("\n" + "=" * 70)
    print("AUTOMATED QUERY TESTING")
    print("=" * 70)

    # Test the first three queries in the dataset
    test_query_ids = list(queries.keys())[:3]

    for query_id in test_query_ids:

        query_text = queries[query_id]

        print(f"\nTEST QUERY #{query_id}: {query_text}")
        print("-" * 70)

        results = ir_system.search(
            query_text,
            top_k=3
        )

        if not results:
            print("No matching documents found.")
            continue

        print(f"Retrieved {len(results)} document(s):\n")

        for rank, result in enumerate(results, start=1):
            print(
                f"Rank {rank} | "
                f"Document ID: {result['doc_id']} | "
                f"Similarity Score: {result['score']:.4f}"
            )

            print(f"Snippet: {result['snippet']}\n")

    # ------------------------------------------------------
    # PART B: SYSTEM EVALUATION
    # ------------------------------------------------------

    print("\n" + "=" * 70)
    print("SYSTEM EVALUATION METRICS")
    print("=" * 70)

    avg_precision, avg_recall = evaluate_system(
        ir_system,
        queries,
        relevance,
        top_k=5
    )

    print(f"Average Precision@5: {avg_precision:.4f}")
    print(f"Average Recall@5:    {avg_recall:.4f}")

    print("=" * 70)
    print("EXECUTION COMPLETED")
    print("=" * 70)


# ==========================================================
# 7. RUN PROGRAM
# ==========================================================

if __name__ == "__main__":
    main()


CISI INFORMATION RETRIEVAL SYSTEM

Dataset folder: C:\Users\khare\OneDrive\Desktop\project\archive

Loading CISI datasets...
Documents loaded: 1460
Queries loaded: 112
Queries with relevance judgments: 76

Building TF-IDF index...
TF-IDF index built successfully.
Total documents indexed: 1460
Unique terms indexed: 6040

AUTOMATED QUERY TESTING

TEST QUERY #1: What problems and concerns are there in making up descriptive titles? What difficulties are involved in automatically retrieving articles from approximate titles? What is the usual relevance of the content of articles to their titles?
----------------------------------------------------------------------
Retrieved 3 document(s):

Rank 1 | Document ID: 722 | Similarity Score: 0.4042
Snippet: Information Transfer Limitations of Titles of Chemical Documents   Some methods of estimating the minimum amounts of information in a document not retrievable through its title are discussed.  An anal...

Rank 2 | Document ID: 429 | Similarity 

In [3]:
import math
import csv


In [ ]:

import os
import re
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ==========================================
# 1. PROJECT CONFIGURATION
# ==========================================

# Your dataset folder path
BASE_DIR = Path(
    r"C:\Users\khare\OneDrive\Desktop\project\archive"
)

# Download required NLTK stopwords
nltk.download("stopwords", quiet=True)

STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()


# ==========================================
# 2. FIND DATASET FILES
# ==========================================

def find_dataset_file(filename):
    """
    Search for a CISI file inside the specified folder
    and its subfolders.
    """
    if not BASE_DIR.exists():
        raise FileNotFoundError(
            f"Dataset path does not exist: {BASE_DIR}"
        )

    for file_path in BASE_DIR.rglob("*"):
        if (
            file_path.is_file()
            and file_path.name.upper() == filename.upper()
        ):
            return file_path

    raise FileNotFoundError(
        f"{filename} was not found inside {BASE_DIR}."
    )


# ==========================================
# 3. DATASET PARSING MODULE
# ==========================================

def parse_cisi_all(filepath):
    """
    Parse CISI.ALL and extract document IDs,
    titles, and abstracts/text.
    """
    documents = {}
    current_doc = None
    current_field = None

    with open(
        filepath, "r", encoding="utf-8", errors="ignore"
    ) as file:

        for line in file:
            line_stripped = line.strip()

            if line_stripped.startswith(".I"):
                if current_doc is not None:
                    doc_id = current_doc["id"]
                    content = (
                        current_doc["title"]
                        + " "
                        + current_doc["text"]
                    ).strip()
                    documents[doc_id] = content

                current_doc = {
                    "id": line_stripped[2:].strip(),
                    "title": "",
                    "text": ""
                }
                current_field = None

            elif current_doc is not None:
                if line_stripped.startswith(".T"):
                    current_field = "title"
                    current_doc["title"] += (
                        line_stripped[2:].strip() + " "
                    )

                elif line_stripped.startswith(".W"):
                    current_field = "text"
                    current_doc["text"] += (
                        line_stripped[2:].strip() + " "
                    )

                elif line_stripped.startswith(
                    (".A", ".B", ".N", ".X")
                ):
                    current_field = None

                elif current_field in ("title", "text"):
                    current_doc[current_field] += (
                        line_stripped + " "
                    )

    if current_doc is not None:
        doc_id = current_doc["id"]
        content = (
            current_doc["title"]
            + " "
            + current_doc["text"]
        ).strip()
        documents[doc_id] = content

    return documents


def parse_cisi_qry(filepath):
    """
    Parse CISI.QRY and extract query IDs and text.
    """
    queries = {}
    current_query = None
    current_field = None

    with open(
        filepath, "r", encoding="utf-8", errors="ignore"
    ) as file:

        for line in file:
            line_stripped = line.strip()

            if line_stripped.startswith(".I"):
                if current_query is not None:
                    queries[current_query["id"]] = (
                        current_query["text"].strip()
                    )

                current_query = {
                    "id": line_stripped[2:].strip(),
                    "text": ""
                }
                current_field = None

            elif current_query is not None:
                if line_stripped.startswith(".W"):
                    current_field = "text"
                    current_query["text"] += (
                        line_stripped[2:].strip() + " "
                    )

                elif line_stripped.startswith(
                    (".T", ".A", ".B", ".N", ".X")
                ):
                    current_field = None

                elif current_field == "text":
                    current_query["text"] += (
                        line_stripped + " "
                    )

    if current_query is not None:
        queries[current_query["id"]] = (
            current_query["text"].strip()
        )

    return queries


def parse_cisi_rel(filepath):
    """
    Parse CISI.REL.

    Each line contains a query ID, a relevant document ID,
    and possibly a relevance score. The first two fields
    are used to create the relevance judgments.
    """
    relevance = defaultdict(set)

    with open(
        filepath, "r", encoding="utf-8", errors="ignore"
    ) as file:

        for line in file:
            parts = line.strip().split()

            if len(parts) >= 2:
                query_id = parts[0]
                document_id = parts[1]
                relevance[query_id].add(document_id)

    return relevance


# ==========================================
# 4. TEXT PREPROCESSING MODULE
# ==========================================

def preprocess_text(text):
    """
    Preprocessing:
    1. Convert to lowercase
    2. Tokenize
    3. Remove English stopwords
    4. Apply Porter stemming
    """
    text = text.lower()

    tokens = re.findall(r"\b[a-z0-9]+\b", text)

    tokens = [
        token for token in tokens
        if token not in STOP_WORDS
    ]

    tokens = [
        STEMMER.stem(token)
        for token in tokens
    ]

    return " ".join(tokens)


# ==========================================
# 5. TF-IDF INDEXING AND RETRIEVAL
# ==========================================

class InformationRetrievalSystem:

    def __init__(self, documents):
        self.documents = documents
        self.doc_ids = list(documents.keys())

        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = None

    def build_index(self):
        """
        Preprocess documents and create the TF-IDF index.
        """
        print("\nBuilding TF-IDF index...")

        processed_docs = [
            preprocess_text(self.documents[doc_id])
            for doc_id in self.doc_ids
        ]

        if not any(text.strip() for text in processed_docs):
            raise ValueError(
                "No searchable text found in the documents."
            )

        self.tfidf_matrix = self.vectorizer.fit_transform(
            processed_docs
        )

        print("TF-IDF index built successfully.")
        print(f"Documents indexed: {len(self.doc_ids)}")
        print(
            f"Unique indexed terms: "
            f"{len(self.vectorizer.vocabulary_)}"
        )

    def search(self, query, top_k=10):
        """
        Retrieve documents ranked by cosine similarity.
        """
        if self.tfidf_matrix is None:
            raise RuntimeError(
                "Build the index before searching."
            )

        processed_query = preprocess_text(query)

        if not processed_query.strip():
            return []

        query_vector = self.vectorizer.transform(
            [processed_query]
        )

        similarities = cosine_similarity(
            query_vector, self.tfidf_matrix
        ).flatten()

        # Sort by score, highest first
        top_indices = np.argsort(similarities)[::-1]

        results = []

        for idx in top_indices:
            score = float(similarities[idx])

            # Ignore documents with no matching terms
            if score <= 0:
                continue

            doc_id = self.doc_ids[idx]
            results.append((doc_id, score))

            if len(results) >= top_k:
                break

        return results


# ==========================================
# 6. EVALUATION METRICS
# ==========================================

def precision_at_k(retrieved_ids, relevant_ids, k):
    """
    Precision@K = relevant retrieved documents / K.
    """
    if k <= 0:
        return 0.0

    relevant_set = set(relevant_ids)
    retrieved_k = retrieved_ids[:k]

    hits = sum(
        1 for doc_id in retrieved_k
        if doc_id in relevant_set
    )

    return hits / k


def recall_at_k(retrieved_ids, relevant_ids, k):
    """
    Recall@K = relevant retrieved documents / all relevant documents.
    """
    relevant_set = set(relevant_ids)

    if not relevant_set or k <= 0:
        return 0.0

    retrieved_k = retrieved_ids[:k]

    hits = sum(
        1 for doc_id in retrieved_k
        if doc_id in relevant_set
    )

    return hits / len(relevant_set)


def average_precision(retrieved_ids, relevant_ids):
    """
    Average Precision for one query.
    """
    relevant_set = set(relevant_ids)

    if not relevant_set:
        return 0.0

    hits = 0
    sum_precision = 0.0

    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_set:
            hits += 1
            sum_precision += hits / rank

    return sum_precision / len(relevant_set)


def ndcg_at_k(retrieved_ids, relevant_ids, k):
    """
    Normalized Discounted Cumulative Gain@K.
    Uses binary relevance judgments.
    """
    if k <= 0:
        return 0.0

    relevant_set = set(relevant_ids)

    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, doc_id in enumerate(
            retrieved_ids[:k], start=1
        )
        if doc_id in relevant_set
    )

    ideal_relevant = min(len(relevant_set), k)

    idcg = sum(
        1.0 / math.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )

    return dcg / idcg if idcg > 0 else 0.0


# ==========================================
# 7. MAIN EXECUTION
# ==========================================

def main():

    print("=" * 65)
    print("      CISI INFORMATION RETRIEVAL SYSTEM")
    print("=" * 65)

    # Locate all dataset files
    doc_path = find_dataset_file("CISI.ALL")
    qry_path = find_dataset_file("CISI.QRY")
    rel_path = find_dataset_file("CISI.REL")

    print("\nDataset files found:")
    print("Documents :", doc_path)
    print("Queries   :", qry_path)
    print("Relevance :", rel_path)

    # Load the actual CISI dataset
    print("\nLoading CISI dataset...")

    documents = parse_cisi_all(doc_path)
    queries = parse_cisi_qry(qry_path)
    relevance = parse_cisi_rel(rel_path)

    print(f"Documents loaded: {len(documents)}")
    print(f"Queries loaded  : {len(queries)}")
    print(f"Queries with relevance judgments: {len(relevance)}")

    if not documents:
        raise ValueError(
            "No documents were parsed from CISI.ALL."
        )

    if not queries:
        raise ValueError(
            "No queries were parsed from CISI.QRY."
        )

    # Build the index
    ir_system = InformationRetrievalSystem(documents)
    ir_system.build_index()

    # Evaluate the first three queries that have relevance judgments
    test_queries = [
        query_id
        for query_id in queries
        if query_id in relevance and relevance[query_id]
    ][:3]

    if not test_queries:
        raise ValueError(
            "No queries with relevance judgments were found. "
            "Check the CISI.REL file."
        )

    k = 3

    all_precision = []
    all_recall = []
    all_ap = []
    all_ndcg = []

    print("\n" + "=" * 65)
    print("          QUERY EVALUATION AND METRICS")
    print("=" * 65)

    for query_id in test_queries:

        query_text = queries[query_id]
        relevant_docs = relevance[query_id]

        print(f"\nQuery ID: {query_id}")
        print("Query:", query_text)
        print("-" * 65)

        # Retrieve up to 10 results for AP calculation
        results = ir_system.search(query_text, top_k=10)

        retrieved_ids = [
            doc_id for doc_id, score in results
        ]

        print("\nTop 3 Retrieved Documents:")

        if not results:
            print("No matching documents found.")

        for rank, (doc_id, score) in enumerate(
            results[:3], start=1
        ):
            label = (
                "RELEVANT"
                if doc_id in relevant_docs
                else "NOT RELEVANT"
            )

            print(
                f"{rank}. Document ID: {doc_id} | "
                f"Score: {score:.4f} | {label}"
            )

            print(
                "   Text:",
                documents[doc_id][:180].replace("\n", " "),
                "..."
            )

        # Calculate evaluation metrics
        precision = precision_at_k(
            retrieved_ids, relevant_docs, k
        )

        recall = recall_at_k(
            retrieved_ids, relevant_docs, k
        )

        ap = average_precision(
            retrieved_ids, relevant_docs
        )

        ndcg = ndcg_at_k(
            retrieved_ids, relevant_docs, k
        )

        print(f"\nEvaluation Metrics @ {k}:")
        print(f"Precision@{k}: {precision:.4f}")
        print(f"Recall@{k}   : {recall:.4f}")
        print(f"Average Precision: {ap:.4f}")
        print(f"nDCG@{k}     : {ndcg:.4f}")

        all_precision.append(precision)
        all_recall.append(recall)
        all_ap.append(ap)
        all_ndcg.append(ndcg)

    # Overall average performance
    print("\n" + "=" * 65)
    print("           OVERALL SYSTEM PERFORMANCE")
    print("=" * 65)

    print(
        f"Mean Precision@{k}: "
        f"{np.mean(all_precision):.4f}"
    )

    print(
        f"Mean Recall@{k}   : "
        f"{np.mean(all_recall):.4f}"
    )

    print(
        f"Mean Average Precision (MAP): "
        f"{np.mean(all_ap):.4f}"
    )

    print(
        f"Mean nDCG@{k}      : "
        f"{np.mean(all_ndcg):.4f}"
    )

    print("=" * 65)

    # ==========================================
    # 8. INTERACTIVE SEARCH
    # ==========================================

    print("\nInteractive Search")
    print("Enter a query to search the CISI documents.")
    print("Type 'exit' to stop.")

    while True:

        user_query = input("\nEnter your query: ").strip()

        if user_query.lower() == "exit":
            print("Search system closed.")
            break

        if not user_query:
            print("Please enter a search query.")
            continue

        results = ir_system.search(user_query, top_k=5)

        if not results:
            print("No matching documents found.")
            continue

        print("\nTop Search Results:")

        for rank, (doc_id, score) in enumerate(
            results, start=1
        ):
            print(f"\n{rank}. Document ID: {doc_id}")
            print(f"   Similarity Score: {score:.4f}")
            print(
                "   Content:",
                documents[doc_id][:300].replace("\n", " "),
                "..."
            )


if __name__ == "__main__":
    main()


      CISI INFORMATION RETRIEVAL SYSTEM

Dataset files found:
Documents : C:\Users\khare\OneDrive\Desktop\project\archive\CISI.ALL
Queries   : C:\Users\khare\OneDrive\Desktop\project\archive\CISI.QRY
Relevance : C:\Users\khare\OneDrive\Desktop\project\archive\CISI.REL

Loading CISI dataset...
Documents loaded: 1460
Queries loaded  : 112
Queries with relevance judgments: 76

Building TF-IDF index...
TF-IDF index built successfully.
Documents indexed: 1460
Unique indexed terms: 6032

          QUERY EVALUATION AND METRICS

Query ID: 1
Query: What problems and concerns are there in making up descriptive titles? What difficulties are involved in automatically retrieving articles from approximate titles? What is the usual relevance of the content of articles to their titles?
-----------------------------------------------------------------

Top 3 Retrieved Documents:
1. Document ID: 722 | Score: 0.4042 | RELEVANT
   Text: Information Transfer Limitations of Titles of Chemical Documents   So